In [ ]:
# Install Unsloth and other required libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Maximum context window
dtype = None # Auto-detect
load_in_4bit = True # Squeezes the model to fit on the free T4 GPU

# Choose your model here! (Uncomment the one you want)
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit" 
# model_name = "unsloth/gemma-2-9b-it-bnb-4bit"

print(f"Loading Base Chef: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model loaded successfully!")

In [ ]:
# 1. Attach the LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank/size of the adapter (16 is perfect for this)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

print("LoRA Adapters successfully attached! The Chef is ready to learn.")

In [ ]:
from datasets import load_dataset

# Load your custom JSONL file
dataset = load_dataset("json", data_files="culinary_philosophy_dataset.jsonl", split="train")

# Define the Prompt Template
# This maps your JSON to a conversation format
prompt_template = """Below is an instruction that describes a culinary task. Write a response that appropriately completes the request, applying Vietnamese Five Elements and French technique.

### Instruction:
{}

### Response:
{}"""

# Function to apply the template to every row in your dataset
def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output)
        texts.append(text)
    return { "text" : texts }

# Process the dataset
dataset = dataset.map(format_prompts, batched = True)

print(f"Dataset formatted! Loaded {len(dataset)} examples.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        # Set max_steps to roughly the size of your dataset (e.g., 60-100)
        # If you have a small dataset, let it run for 60 steps to learn deeply
        max_steps = 60, 
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting the training process...")
trainer_stats = trainer.train()
print("🎉 Training Complete! You have a custom Master Chef AI.")

In [ ]:
model.push_to_hub("your-huggingface-username/cordon-bleu-yinyang-chef", token = "YOUR_HF_TOKEN")
tokenizer.push_to_hub("your-huggingface-username/cordon-bleu-yinyang-chef", token = "YOUR_HF_TOKEN")